# 15 -- Multi-Tenant ML
**ServiceTitan context**: ~12,000 contractor tenants share one platform. A 2-tech plumbing shop and a 200-tech HVAC enterprise need the same models to work well. This notebook covers global models, per-tenant fine-tuning, cold start, data isolation, and personalization.

Topics: Global vs per-tenant models * Cold start with priors * Tenant embeddings * Hierarchical models * Privacy-preserving ML

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

## 1. Simulate Multi-Tenant Data

In [2]:
# Generate synthetic contractors with different business profiles
N_TENANTS = 50
N_JOBS_PER_TENANT = (20, 2000)  # small shops vs large enterprises

tenant_profiles = []
for tid in range(N_TENANTS):
    size = np.random.choice(['small','medium','large'], p=[0.5, 0.35, 0.15])
    n_jobs = {'small': np.random.randint(20,150),
              'medium': np.random.randint(150,600),
              'large': np.random.randint(600,2000)}[size]
    # Each tenant has different base conversion rates (their market, their techs)
    base_rate = np.random.beta(3, 5)   # true close rate ~ 0.2-0.6
    avg_job_val = np.random.normal(800, 200) if size=='small' else np.random.normal(1200, 400)
    tenant_profiles.append({'tid': tid, 'size': size, 'n_jobs': n_jobs,
                            'base_rate': base_rate, 'avg_val': max(200, avg_job_val)})

def make_tenant_jobs(profile):
    n = profile['n_jobs']
    X = np.column_stack([
        np.random.normal(profile['avg_val'], 300, n),  # job_value
        np.random.uniform(0, 1, n),                    # urgency
        np.random.binomial(1, 0.3, n),                 # returning_customer
        np.random.normal(3, 1, n).clip(1, 10),         # options_presented
    ])
    # True close probability depends on features + tenant-specific intercept
    logit = (-2.0 + 0.001*X[:,0] + 0.8*X[:,1] + 0.5*X[:,2] + 0.2*X[:,3]
             + np.log(profile['base_rate']/(1-profile['base_rate']+1e-9)))
    y = (np.random.random(n) < 1/(1+np.exp(-logit))).astype(int)
    return X, y

all_data = [make_tenant_jobs(p) for p in tenant_profiles]
sizes = [p['n_jobs'] for p in tenant_profiles]
print(f'Tenants: {N_TENANTS}')
print(f'Jobs: min={min(sizes)}, median={int(np.median(sizes))}, max={max(sizes)}')
print(f'Small: {sum(1 for p in tenant_profiles if p["size"]=="small")}  '
      f'Medium: {sum(1 for p in tenant_profiles if p["size"]=="medium")}  '
      f'Large: {sum(1 for p in tenant_profiles if p["size"]=="large")}')

Tenants: 50
Jobs: min=27, median=162, max=1965
Small: 25  Medium: 12  Large: 13


## 2. Global Model (One Model for All Tenants)

In [3]:
# Pool all tenant data and train one model
# Pro: lots of data, handles cold start
# Con: ignores tenant-specific patterns

X_all = np.vstack([x for x,_ in all_data])
y_all = np.concatenate([y for _,y in all_data])
tenant_ids = np.concatenate([[i]*len(all_data[i][1]) for i in range(N_TENANTS)])

scaler = StandardScaler().fit(X_all)
X_all_s = scaler.transform(X_all)

global_model = LogisticRegression(max_iter=500)
global_model.fit(X_all_s, y_all)

# Evaluate per-tenant accuracy
global_accs = []
for i in range(N_TENANTS):
    mask = tenant_ids == i
    if mask.sum() < 10: continue
    acc = global_model.score(X_all_s[mask], y_all[mask])
    global_accs.append(acc)

print(f'Global model performance:')
print(f'  Mean accuracy: {np.mean(global_accs):.3f}')
print(f'  Std accuracy:  {np.std(global_accs):.3f}')
print(f'  Min accuracy:  {np.min(global_accs):.3f}  (worst-served tenant)')
print(f'  Max accuracy:  {np.max(global_accs):.3f}')

Global model performance:
  Mean accuracy: 0.639
  Std accuracy:  0.113
  Min accuracy:  0.407  (worst-served tenant)
  Max accuracy:  0.912


## 3. Per-Tenant Fine-Tuning (Warm Start)

In [4]:
# Start from global model, fine-tune on tenant's own data
# Key: requires enough data per tenant (cold start problem for new tenants)

per_tenant_accs = []
cold_start_accs = []

for i, (X_t, y_t) in enumerate(all_data):
    if len(y_t) < 10: continue
    X_s = scaler.transform(X_t)
    
    # Only fine-tune if enough data (>50 jobs)
    if len(y_t) >= 50:
        # Warm-start: initialize from global model coefficients
        local_model = LogisticRegression(max_iter=200, warm_start=True, C=1.0)
        local_model.coef_     = global_model.coef_.copy()
        local_model.intercept_= global_model.intercept_.copy()
        local_model.classes_  = global_model.classes_.copy()
        local_model.fit(X_s, y_t)
        per_tenant_accs.append(local_model.score(X_s, y_t))
    else:
        # Cold start: fall back to global model
        cold_start_accs.append(global_model.score(X_s, y_t))

print(f'Per-tenant fine-tuned ({len(per_tenant_accs)} tenants with >50 jobs):')
print(f'  Mean: {np.mean(per_tenant_accs):.3f}  Std: {np.std(per_tenant_accs):.3f}')
print(f'Cold start / global fallback ({len(cold_start_accs)} small tenants):')
print(f'  Mean: {np.mean(cold_start_accs):.3f}  Std: {np.std(cold_start_accs):.3f}')

Per-tenant fine-tuned (44 tenants with >50 jobs):
  Mean: 0.694  Std: 0.090
Cold start / global fallback (6 small tenants):
  Mean: 0.660  Std: 0.103


## 4. Hierarchical Model (Industry Priors)

In [5]:
# Bayesian hierarchical: tenant parameters drawn from industry distribution
# theta_tenant ~ N(theta_global, sigma^2)
# This is the statistically principled answer to cold start

# Practical approximation: regularize toward global model estimate
# Strong regularization (high C inverse) = closer to global; weak = more personalized

def hierarchical_model(X_t, y_t, X_global, y_global, tenant_weight=0.3):
    # Blend tenant data with downsampled global data
    n_global = max(len(y_t), 100)  # at least as many global as tenant
    idx = np.random.choice(len(y_global), min(n_global, len(y_global)), replace=False)
    X_blend = np.vstack([X_t, X_global[idx] * (1-tenant_weight)])
    y_blend = np.concatenate([y_t, y_global[idx]])
    m = LogisticRegression(max_iter=300, C=0.5)
    m.fit(X_blend, y_blend)
    return m

# Evaluate for small tenants where this matters most
hier_accs, global_fallback_accs = [], []
for i in range(N_TENANTS):
    X_t, y_t = all_data[i]
    if len(y_t) > 100: continue  # only care about small tenant case
    X_s = scaler.transform(X_t)
    hier_m = hierarchical_model(X_s, y_t, X_all_s, y_all)
    hier_accs.append(hier_m.score(X_s, y_t))
    global_fallback_accs.append(global_model.score(X_s, y_t))

print(f'Small tenant performance comparison (n={len(hier_accs)} tenants <100 jobs):')
print(f'  Global fallback:   mean={np.mean(global_fallback_accs):.3f}')
print(f'  Hierarchical:      mean={np.mean(hier_accs):.3f}  (blends industry priors with tenant data)')

Small tenant performance comparison (n=15 tenants <100 jobs):
  Global fallback:   mean=0.694
  Hierarchical:      mean=0.711  (blends industry priors with tenant data)


## 5. Data Isolation: The Multi-Tenant Privacy Contract

In [6]:
# ServiceTitan must guarantee tenant A's pricing/customer data never affects
# tenant B's model predictions.

# ISOLATION STRATEGIES:
# 1. Global model: safe -- only aggregate statistics, no per-tenant leakage
#    Risk: model might memorize small tenant patterns during training
# 2. Per-tenant model: safe -- completely separate models, separate storage
#    Risk: requires secure model registry (each tenant can only load their own)
# 3. Federated learning: train locally per tenant, aggregate only gradients
#    Risk: gradient inversion attacks; complex infra
# 4. Differential privacy (DP): add calibrated noise to training gradients
#    Risk: accuracy degradation at small epsilon (strong privacy)

# DP simulation: epsilon = privacy budget; lower = more private, less accurate
from sklearn.linear_model import SGDClassifier

epsilons = [0.1, 0.5, 1.0, 2.0, 5.0, float('inf')]  # inf = no DP
dp_accs  = []

X_demo, y_demo = all_data[0]
X_demo_s = scaler.transform(X_demo)

for eps in epsilons:
    m = SGDClassifier(loss='log_loss', max_iter=300, random_state=42)
    noise_scale = 1.0 / eps if eps < float('inf') else 0.0
    X_noisy = X_demo_s + np.random.laplace(0, noise_scale, X_demo_s.shape)
    m.fit(X_noisy, y_demo)
    dp_accs.append(m.score(X_demo_s, y_demo))

print('Privacy-Accuracy Tradeoff (Simulated Differential Privacy):')
print(f'  {"Epsilon":<12} {"Accuracy":<12} {"Privacy"}')
print('-'*40)
for eps, acc in zip(epsilons, dp_accs):
    priv = 'Very high' if eps<0.5 else 'High' if eps<1 else 'Medium' if eps<3 else 'Low' if eps<float('inf') else 'None (baseline)'
    print(f'  {str(eps):<12} {acc:.3f}        {priv}')

Privacy-Accuracy Tradeoff (Simulated Differential Privacy):
  Epsilon      Accuracy     Privacy
----------------------------------------
  0.1          0.839        Very high
  0.5          0.839        High
  1.0          0.839        Medium
  2.0          0.777        Medium
  5.0          0.839        Low
  inf          0.795        None (baseline)


## 6. Cold Start: New Tenant Onboarding Strategy

In [7]:
# New tenant = 0 jobs. Can't train a model. How do we serve them?

def cold_start_strategy(n_jobs_so_far: int, vertical: str, region: str):
    if n_jobs_so_far == 0:
        strategy = 'global_prior'
        confidence = 'low'
        model_source = f'Global model (all {N_TENANTS} tenants)'
    elif n_jobs_so_far < 20:
        strategy = 'vertical_prior'
        confidence = 'low-medium'
        model_source = f'Vertical model ({vertical} contractors, similar size)'
    elif n_jobs_so_far < 100:
        strategy = 'hierarchical'
        confidence = 'medium'
        model_source = f'Hierarchical (30% tenant data + 70% vertical prior)'
    elif n_jobs_so_far < 500:
        strategy = 'warm_start_finetuned'
        confidence = 'high'
        model_source = f'Fine-tuned from global (80% tenant data)'
    else:
        strategy = 'fully_personalized'
        confidence = 'very high'
        model_source = f'Fully tenant-specific model ({n_jobs_so_far} jobs)'
    return strategy, confidence, model_source

print('Cold Start Decision Policy:')
print(f'  {"Jobs Completed":<20} {"Strategy":<25} {"Confidence":<15} Model Source')
print('-'*90)
for n in [0, 5, 50, 200, 1000]:
    strat, conf, src = cold_start_strategy(n, 'HVAC', 'Southwest')
    print(f'  {n:<20} {strat:<25} {conf:<15} {src}')

Cold Start Decision Policy:
  Jobs Completed       Strategy                  Confidence      Model Source
------------------------------------------------------------------------------------------
  0                    global_prior              low             Global model (all 50 tenants)
  5                    vertical_prior            low-medium      Vertical model (HVAC contractors, similar size)
  50                   hierarchical              medium          Hierarchical (30% tenant data + 70% vertical prior)
  200                  warm_start_finetuned      high            Fine-tuned from global (80% tenant data)
  1000                 fully_personalized        very high       Fully tenant-specific model (1000 jobs)


## 7. Atlas Personalization -- Per-Tenant Context

In [8]:
# Atlas needs per-tenant context for grounded responses:
# 'How many jobs did we complete this week?' requires tenant-specific data
# Three approaches to per-tenant personalization in LLM context:

approaches = {
    'Fine-tuning per tenant': {
        'cost':   'Very high (full training run per tenant)',
        'latency':'Deployment latency (load per-tenant weights)',
        'quality':'Highest -- full adaptation',
        'when':   'Enterprise tier, >10k jobs, custom behavior needed',
    },
    'RAG (tenant data in context)': {
        'cost':   'Low (retrieval + inference only)',
        'latency':'Adds 10-50ms for retrieval',
        'quality':'Good -- grounded in real tenant data',
        'when':   'Most tenants; data in vector store scoped by tenant_id',
    },
    'Prompt injection (stats in system prompt)': {
        'cost':   'Very low',
        'latency':'Adds tokens (cost) but no extra call',
        'quality':'OK for simple stats; fails for complex history',
        'when':   'Dashboard summaries, basic Q&A on recent KPIs',
    },
    'In-context learning examples': {
        'cost':   'Medium (tokens)',
        'latency':'Linear in examples',
        'quality':'Good for few-shot adaptation',
        'when':   'Estimate drafting with tenant-specific pricing style',
    },
}

print('Atlas Personalization Strategies:')
print('='*70)
for approach, props in approaches.items():
    print(f'\n  {approach}')
    for k,v in props.items():
        print(f'    {k:<12}: {v}')

Atlas Personalization Strategies:

  Fine-tuning per tenant
    cost        : Very high (full training run per tenant)
    latency     : Deployment latency (load per-tenant weights)
    quality     : Highest -- full adaptation
    when        : Enterprise tier, >10k jobs, custom behavior needed

  RAG (tenant data in context)
    cost        : Low (retrieval + inference only)
    latency     : Adds 10-50ms for retrieval
    quality     : Good -- grounded in real tenant data
    when        : Most tenants; data in vector store scoped by tenant_id

  Prompt injection (stats in system prompt)
    cost        : Very low
    latency     : Adds tokens (cost) but no extra call
    quality     : OK for simple stats; fails for complex history
    when        : Dashboard summaries, basic Q&A on recent KPIs

  In-context learning examples
    cost        : Medium (tokens)
    latency     : Linear in examples
    quality     : Good for few-shot adaptation
    when        : Estimate drafting with t